# Machine Learning e Visão Computacional [T2]
## Mini-Projeto Avaliativo - Módulo 1 - Semana 07

### Pipeline de ETL (Extract, Transform, Load) para Dados da Olist

Este notebook executa o tratamento dos datasets `olist_products_dataset.csv` e `olist_orders_dataset.csv`, implementando as lógicas de negócio requeridas sem a utilização de bibliotecas como Pandas.

### 1. Reflexão Teórica: Machine Learning e Qualidade dos Dados

Uma lógica de programação aplicada à limpeza correta dos dados (ETL) é a principal defesa contra o princípio fundamental do *Garbage In, Garbage Out*. Ao lidar com atributos inconsistentes — como categorias com caracteres especiais soltos, registros parcialmente nulos ou formatos de datas incorretos — evitamos que os algoritmos de Machine Learning interpretem esses erros como padrões relevantes. Quando modelos estatísticos ou de Deep Learning treinam sobre um *dataset* despadronizado, as correlações aprendidas tornam-se ruidosas. Por exemplo, uma mesma categoria escrita como "Móveis!" e "moveis" será lida pelo modelo como duas *features* distintas, fragmentando o poder preditivo e dispersando pesos que deveriam ser unificados.

Além disso, o tratamento consciente dos registros nulos, como será feito abaixo com a imputação de `0.0` nas dimensões físicas, ajuda a prevenir vieses severos. Se descartássemos as linhas de produtos sem dimensões físicas sem critério, poderíamos inadvertidamente remover classes inteiras de produtos digitais ou imateriais, causando **Overfitting** nas classes restantes. O modelo final tornar-se-ia "viciado" em predizer e focar apenas em produtos físicos superespecificados, perdendo a generalização.

In [1]:
import csv
import re
import os
from datetime import datetime

### 2. Funções de Limpeza (Sanitização e Regras de Negócio)

Abaixo definimos a limpeza das strings de categoria e a nossa regra de corte para as dimensões nulas.

In [2]:
def limpar_categoria(categoria):
    """
    Padroniza a string da categoria utilizando métodos nativos de string e RegEx.
    """
    if not categoria:
        return "Sem Categoria"
    
    # 1. Converter para letras minúsculas e remover espaços excedentes (início e fim)
    categoria = categoria.lower().strip()
    
    # 2. Utilizar Regex para remover caracteres especiais ou pontuações indevidas
    categoria_limpa = re.sub(r'[^\w\s]', '', categoria)
    
    # Após a remoção, é possível que sobrem espaços extras ou a string fique vazia
    categoria_limpa = categoria_limpa.strip()
    
    if not categoria_limpa:
        return "Sem Categoria"
        
    return categoria_limpa

def tratar_dimensoes_fisicas(valor):
    """
    Regra de Corte: Se os valores das dimensões físicas ou peso forem vazios,
    atribuiremos o valor padrão de '0.0'.
    """
    if not valor or valor.strip() == '':
        return '0.0'
    return valor

### 3. Funções de Processamento de Arquivos
As funções a seguir leem os arquivos `.csv` e aplicam as transformações linha a linha, **salvando simultaneamente** os dados tratados no novo arquivo.

In [3]:
def processar_produtos(filepath, output_path):
    """
    Lê o arquivo de produtos, aplica sanitização e salva o resultado no output_path.
    """
    linhas_processadas = 0
    nulos_corrigidos = 0
    colunas_dimensoes = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

    with open(filepath, mode='r', encoding='utf-8') as file, \
         open(output_path, mode='w', encoding='utf-8', newline='') as out_file:
         
        reader = csv.DictReader(file)
        writer = csv.DictWriter(out_file, fieldnames=reader.fieldnames)
        writer.writeheader()
        
        for row in reader:
            linhas_processadas += 1
            
            categoria_original = row.get('product_category_name', '')
            categoria_limpa = limpar_categoria(categoria_original)
            
            if not categoria_original or categoria_limpa == "Sem Categoria" and categoria_original != "Sem Categoria":
                if not categoria_original:
                    nulos_corrigidos += 1
            
            row['product_category_name'] = categoria_limpa
            
            for col in colunas_dimensoes:
                valor_original = row.get(col, '')
                if not valor_original or valor_original.strip() == '':
                    nulos_corrigidos += 1
                row[col] = tratar_dimensoes_fisicas(valor_original)
                
            writer.writerow(row)
            
    return linhas_processadas, nulos_corrigidos

def processar_pedidos(filepath, output_path):
    """
    Lê o arquivo de pedidos, formata datas, coleta estatísticas e salva no output_path.
    """
    linhas_processadas = 0
    pedidos_cancelados = 0
    entregas_vazias_total = 0
    entregas_vazias_e_canceladas = 0
    entregas_vazias_nao_canceladas = 0
    
    with open(filepath, mode='r', encoding='utf-8') as file, \
         open(output_path, mode='w', encoding='utf-8', newline='') as out_file:
         
        reader = csv.DictReader(file)
        writer = csv.DictWriter(out_file, fieldnames=reader.fieldnames)
        writer.writeheader()
        
        for row in reader:
            linhas_processadas += 1
            
            status = row.get('order_status', '')
            data_entrega = row.get('order_delivered_customer_date', '')
            data_aprovacao = row.get('order_approved_at', '')
            
            if not data_entrega or data_entrega.strip() == '':
                entregas_vazias_total += 1
                if status == 'canceled':
                    entregas_vazias_e_canceladas += 1
                else:
                    entregas_vazias_nao_canceladas += 1
            
            if status == 'canceled':
                pedidos_cancelados += 1
                
            if data_aprovacao and data_aprovacao.strip() != '':
                try:
                    data_obj = datetime.strptime(data_aprovacao, "%Y-%m-%d %H:%M:%S")
                    row['order_approved_at'] = data_obj.strftime("%d/%m/%Y")
                except ValueError:
                    pass
                    
            writer.writerow(row)

    hipotese_comprovada = (entregas_vazias_nao_canceladas == 0) and (entregas_vazias_total > 0)
    return linhas_processadas, pedidos_cancelados, entregas_vazias_total, entregas_vazias_e_canceladas, entregas_vazias_nao_canceladas, hipotese_comprovada

### 4. Execução do Pipeline e Relatório de Status
Orquestra os caminhos e invoca as funções que tratam e gravam os novos arquivos CSV.

In [4]:
produtos_path = os.path.join("data_repo", "olist_products_dataset.csv")
produtos_out = os.path.join("data_repo", "olist_products_dataset_tratado.csv")

pedidos_path = os.path.join("data_repo", "olist_orders_dataset.csv")
pedidos_out = os.path.join("data_repo", "olist_orders_dataset_tratado.csv")

print("Iniciando o processamento do pipeline de ETL...")

# Processando produtos
if os.path.exists(produtos_path):
    linhas_prod, nulos_prod = processar_produtos(produtos_path, produtos_out)
    print(f"-> Arquivo Produtos TRATADO salvo em: {produtos_out}")
else:
    print(f"Erro: Arquivo não encontrado - {produtos_path}")
    linhas_prod, nulos_prod = 0, 0

# Processando pedidos
if os.path.exists(pedidos_path):
    linhas_ped, ped_canc, entr_vaz_tot, entr_vaz_canc, entr_vaz_ncanc, hip_comprovada = processar_pedidos(pedidos_path, pedidos_out)
    print(f"-> Arquivo Pedidos TRATADO salvo em: {pedidos_out}\n")
else:
    print(f"Erro: Arquivo não encontrado - {pedidos_path}")
    linhas_ped, ped_canc, entr_vaz_tot, entr_vaz_canc, entr_vaz_ncanc, hip_comprovada = 0, 0, 0, 0, 0, False

print("="*60)
print("         RELATÓRIO DE STATUS - ETL OLIST")
print("="*60)
print(f"-> Arquivo de Produtos Processado:")
print(f"   - Linhas processadas: {linhas_prod}")
print(f"   - Total de registros nulos corrigidos: {nulos_prod}\n")
print(f"-> Arquivo de Pedidos Processado:")
print(f"   - Linhas processadas: {linhas_ped}")
print(f"   - Pedidos cancelados identificados: {ped_canc}\n")
print("-> Validação da Hipótese de Negócio (Diretoria):")
print(f"   - 'Datas de entrega estão nulas obrigatoriamente porque o pedido foi cancelado?'")
print(f"   - Total de datas de entrega nulas: {entr_vaz_tot}")
print(f"   - Nulas e CANCELADAS: {entr_vaz_canc}")
print(f"   - Nulas e NÃO CANCELADAS: {entr_vaz_ncanc}\n")

if hip_comprovada:
    print("   [CONCLUSÃO]: A hipótese da diretoria está CORRETA.")
else:
    print("   [CONCLUSÃO]: A hipótese da diretoria está INCORRETA.")
    print("   Existem pedidos com datas de entrega nulas que possuem outros status.")
print("="*60)

Iniciando o processamento do pipeline de ETL...
-> Arquivo Produtos TRATADO salvo em: data_repo\olist_products_dataset_tratado.csv
-> Arquivo Pedidos TRATADO salvo em: data_repo\olist_orders_dataset_tratado.csv

         RELATÓRIO DE STATUS - ETL OLIST
-> Arquivo de Produtos Processado:
   - Linhas processadas: 32951
   - Total de registros nulos corrigidos: 618

-> Arquivo de Pedidos Processado:
   - Linhas processadas: 99441
   - Pedidos cancelados identificados: 625

-> Validação da Hipótese de Negócio (Diretoria):
   - 'Datas de entrega estão nulas obrigatoriamente porque o pedido foi cancelado?'
   - Total de datas de entrega nulas: 2965
   - Nulas e CANCELADAS: 619
   - Nulas e NÃO CANCELADAS: 2346

   [CONCLUSÃO]: A hipótese da diretoria está INCORRETA.
   Existem pedidos com datas de entrega nulas que possuem outros status.
